# MedAssist - MedGemma Colab (Ordered Notebook)\nRun cells in order. This notebook is structured for competition demo reliability.

## 1) Install Dependencies\nAfter this cell finishes, restart runtime once before continuing.

In [ ]:
!nvidia-smi || true\n!pip -q uninstall -y transformers\n!pip -q install "transformers==4.57.1" "accelerate>=1.10.0" "bitsandbytes>=0.47.0" "sentencepiece" "huggingface_hub" "gradio"

## 2) Restart Runtime\nMenu: Runtime -> Restart session\nThen continue from next cell.

## 3) Hugging Face Login

In [ ]:
from huggingface_hub import login\nfrom getpass import getpass\n\ntoken = getpass('HF token: ')\nlogin(token=token)\nprint('HF login complete')

## 4) Load MedGemma (4-bit)

In [ ]:
import torch\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline\n\nMODEL_ID = 'google/medgemma-4b-it'\n\nbnb_config = BitsAndBytesConfig(\n    load_in_4bit=True,\n    bnb_4bit_quant_type='nf4',\n    bnb_4bit_compute_dtype=torch.float16,\n    bnb_4bit_use_double_quant=True,\n)\n\ntokenizer = AutoTokenizer.from_pretrained(MODEL_ID)\nmodel = AutoModelForCausalLM.from_pretrained(\n    MODEL_ID,\n    device_map='auto',\n    quantization_config=bnb_config,\n    torch_dtype=torch.float16,\n)\n\ngen = pipeline('text-generation', model=model, tokenizer=tokenizer)\nprint('Loaded:', MODEL_ID)

## 5) Prompt + Helpers + Robust run_case

In [ ]:
import json, re, time

SYSTEM_PROMPT = """You are MedAssist, an AI clinical triage decision-support assistant for low-resource settings.

Safety rules:
1) This is decision support only and not a diagnosis.
2) Never provide medication dosages.
3) Never claim certainty or definitive diagnosis.
4) Prioritize life-threatening red flags.
5) Return ONLY one valid JSON object, no markdown, no extra prose.

Required JSON keys:
triage_summary, differential_diagnosis, urgency_level, red_flags,
recommended_next_steps, patient_friendly_explanation, limitations, disclaimer
"""

def build_prompt(payload):
    return f"""<system>
{SYSTEM_PROMPT}
</system>
<user>
Patient context:
{json.dumps(payload, indent=2)}

Return strict JSON only.
</user>
<assistant>"""

def extract_json(text):
    fence = re.search(r"```json\s*([\s\S]*?)```", text, re.IGNORECASE)
    if fence:
        return json.loads(fence.group(1).strip())

    candidates = re.findall(r"\{[\s\S]*\}", text)
    for c in reversed(candidates):
        try:
            obj = json.loads(c)
            if isinstance(obj, dict):
                return obj
        except:
            pass
    raise ValueError("No JSON found")

def parse_systolic(bp):
    try:
        return int(str(bp).split('/')[0].strip())
    except:
        return -1

def deterministic_red_flags(payload):
    chief = str(payload.get('chief_complaint','')).lower()
    symptoms = str(payload.get('symptoms','')).lower()
    vitals = payload.get('vitals', {})
    spo2 = int(vitals.get('oxygen_saturation', 0) or 0)
    rr = int(vitals.get('respiratory_rate', 0) or 0)
    hr = int(vitals.get('heart_rate', 0) or 0)
    temp = float(vitals.get('temperature', 0) or 0)
    systolic = parse_systolic(vitals.get('blood_pressure', ''))

    flags = []
    if 'chest pain' in chief or 'chest pain' in symptoms:
        flags.append('Possible cardiac ischemia (chest pain pattern)')
    if spo2 < 94:
        flags.append('Low oxygen saturation')
    if 'slurred speech' in symptoms or 'facial droop' in symptoms or 'weakness' in symptoms:
        flags.append('Possible acute stroke signs')
    if rr >= 30 or ('shortness of breath' in symptoms and spo2 < 94):
        flags.append('Marked respiratory distress')
    if 'confusion' in symptoms:
        flags.append('Acute confusion in high-risk patient')
    if 'trauma' in chief or 'injury' in chief or 'road traffic' in chief:
        flags.append('Recent trauma mechanism')
    if systolic != -1 and systolic <= 90:
        flags.append('Hypotension')
    if hr >= 130:
        flags.append('Severe tachycardia')
    if temp >= 39.5:
        flags.append('High fever with potential systemic illness')

    if not flags:
        flags.append('No immediate critical red flags identified from provided data')
    return flags

def fallback_structured(payload, note):
    urg = 'EMERGENCY' if 'chest pain' in str(payload.get('chief_complaint','')).lower() else 'HIGH'
    return {
        'triage_summary': f'Model output parsing failed. {note}',
        'differential_diagnosis': [
            {'condition': 'Acute cardiopulmonary concern', 'rationale': 'Symptoms require urgent evaluation.', 'confidence': 'medium'},
            {'condition': 'Acute coronary syndrome possibility', 'rationale': 'Pattern is concerning for serious etiology.', 'confidence': 'medium'},
            {'condition': 'Other non-cardiac cause', 'rationale': 'Further clinical workup is required.', 'confidence': 'low'}
        ],
        'urgency_level': urg,
        'red_flags': deterministic_red_flags(payload),
        'recommended_next_steps': [
            'Immediate clinician assessment',
            'Repeat vitals and focused exam',
            'Urgent referral/transfer if indicated'
        ],
        'patient_friendly_explanation': (
            'Your symptoms may be serious. Please seek urgent medical care now.'
            if payload.get('patient_friendly_mode') else 'Not requested.'
        ),
        'limitations': 'Generated with safety fallback due to non-JSON model output.',
        'disclaimer': 'This tool is for decision support only and does not replace professional medical judgment.'
    }

def run_case(payload):
    prompt = build_prompt(payload) + "\n\nSTRICT RULE: Output only one JSON object. No markdown."
    t0 = time.perf_counter()

    out = gen(
        prompt,
        do_sample=False,
        max_new_tokens=220,
        return_full_text=False
    )[0]['generated_text']

    parsed_by_model = True
    try:
        parsed = extract_json(out)
    except Exception:
        retry_prompt = prompt + "\nReturn ONLY valid JSON object with required keys. Nothing else."
        out2 = gen(
            retry_prompt,
            do_sample=False,
            max_new_tokens=320,
            return_full_text=False
        )[0]['generated_text']
        try:
            parsed = extract_json(out2)
        except Exception:
            parsed_by_model = False
            parsed = fallback_structured(payload, 'No JSON produced by model after retries.')

    # Always harden red flags with deterministic signals
    merged = []
    seen = set()
    for rf in (parsed.get('red_flags') or []) + deterministic_red_flags(payload):
        t = str(rf).strip()
        k = t.lower()
        if t and k not in seen:
            seen.add(k)
            merged.append(t)
    parsed['red_flags'] = merged

    parsed['inference_time_seconds'] = round(time.perf_counter() - t0, 3)
    parsed['_parsed_by_model'] = parsed_by_model
    return parsed



## 6) Single Sample Case

In [ ]:
sample_case = {\n  "age": 54,\n  "gender": "Male",\n  "chief_complaint": "Severe chest pain",\n  "symptoms": "Crushing central chest pain radiating to left arm, sweating, mild shortness of breath",\n  "duration": "40 minutes",\n  "vitals": {\n    "temperature": 36.9,\n    "heart_rate": 118,\n    "blood_pressure": "160/95",\n    "respiratory_rate": 24,\n    "oxygen_saturation": 93\n  },\n  "medical_history": "Hypertension, smoker",\n  "medications": "Amlodipine",\n  "patient_friendly_mode": True\n}\n\nresult = run_case(sample_case)\nresult

In [ ]:
import json\nwith open('sample_output.json', 'w') as f:\n    json.dump(result, f, indent=2)\nprint('saved sample_output.json')

## 7) Upload test_cases.json

In [ ]:
from google.colab import files\nuploaded = files.upload()\nprint(uploaded.keys())

In [ ]:
import pathlib\ntc_path = pathlib.Path('test_cases.json')\nprint('exists:', tc_path.exists(), '| path:', tc_path.resolve())

## 8) 10-Case Evaluation

In [ ]:
import json, pathlib\n\ncases = json.loads(pathlib.Path('test_cases.json').read_text(encoding='utf-8'))\n\nurgency_correct = 0\ntimes = []\nparsed_direct = 0\n\nfor c in cases:\n    out = run_case(c['input'])\n    pred = out.get('urgency_level')\n    exp = c.get('expected_urgency')\n    ok = (pred == exp)\n    urgency_correct += int(ok)\n    parsed_direct += int(bool(out.get('_parsed_by_model')))\n    times.append(out.get('inference_time_seconds', 0.0))\n    print(f"{c['id']} | pred={pred} expected={exp} | urgency_ok={ok} | parsed_by_model={out.get('_parsed_by_model')} | t={out.get('inference_time_seconds')}s")\n\ntotal = len(cases)\nprint('\n--- Summary ---')\nprint(f"Total cases: {total}")\nprint(f"Urgency correctness: {urgency_correct}/{total} ({urgency_correct/total:.2%})")\nprint(f"Parsed directly as JSON: {parsed_direct}/{total} ({parsed_direct/total:.2%})")\nprint(f"Fallback parser rescue: {total-parsed_direct}/{total} ({(total-parsed_direct)/total:.2%})")\nprint(f"Average inference time: {sum(times)/total:.3f}s")

## 9) Optional Gradio Demo

In [ ]:
import gradio as gr\n\ndef triage_demo(age, gender, chief, symptoms, duration, temp, hr, bp, rr, spo2, history, meds, pf):\n    payload = {\n        "age": int(age),\n        "gender": gender,\n        "chief_complaint": chief,\n        "symptoms": symptoms,\n        "duration": duration,\n        "vitals": {\n            "temperature": float(temp),\n            "heart_rate": int(hr),\n            "blood_pressure": bp,\n            "respiratory_rate": int(rr),\n            "oxygen_saturation": int(spo2),\n        },\n        "medical_history": history,\n        "medications": meds,\n        "patient_friendly_mode": bool(pf),\n    }\n    out = run_case(payload)\n    return json.dumps(out, indent=2)\n\ndemo = gr.Interface(\n    fn=triage_demo,\n    inputs=[\n        gr.Number(value=54, label='Age'),\n        gr.Textbox(value='Male', label='Gender'),\n        gr.Textbox(value='Severe chest pain', label='Chief Complaint'),\n        gr.Textbox(value='Crushing central chest pain radiating to left arm, sweating, mild shortness of breath', label='Symptoms', lines=3),\n        gr.Textbox(value='40 minutes', label='Duration'),\n        gr.Number(value=36.9, label='Temperature'),\n        gr.Number(value=118, label='Heart Rate'),\n        gr.Textbox(value='160/95', label='Blood Pressure'),\n        gr.Number(value=24, label='Respiratory Rate'),\n        gr.Number(value=93, label='Oxygen Saturation'),\n        gr.Textbox(value='Hypertension, smoker', label='Medical History'),\n        gr.Textbox(value='Amlodipine', label='Current Medications'),\n        gr.Checkbox(value=True, label='Patient-Friendly Mode'),\n    ],\n    outputs=gr.Code(language='json', label='Structured Triage Output'),\n    title='MedAssist - MedGemma Colab Demo',\n)\ndemo.launch(share=True)